In [ ]:
# ==========================================
# 📦 CELL 1 — INSTALL DEPENDENCIES
# ==========================================
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.51.0' 'diffusers>=0.32.0' 'accelerate>=0.30.0' 'safetensors'
!pip install -q fastapi uvicorn nest-asyncio python-multipart
!npm install -g localtunnel &> /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb &> /dev/null
print("✅ Cloudflare Tunnel is ready.")

print('✅ All dependencies installed.')

In [ ]:
# ==========================================
# ⚡ CELL 2 — PRE-LOAD AI ENGINE (MAX PERFORMANCE)
# ==========================================
import torch, gc
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler

# Clear VRAM junk
gc.collect()
torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hardware level optimizations
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

IMG_MODEL_ID = 'John6666/janku-v5-nsfw-trained-noobai-rou-wei-illustrious-xl-v50-sdxl'
LORA_MODEL_ID = 'ByteDance/SDXL-Lightning'

print(f'⏳ Loading Engine into VRAM...')
loaded_model = StableDiffusionXLPipeline.from_pretrained(
    IMG_MODEL_ID, torch_dtype=torch.float16, use_safetensors=True
).to(DEVICE)

# Channels Last = massive speedup for T4 GPUs
loaded_model.unet.to(memory_format=torch.channels_last)

loaded_model.scheduler = EulerDiscreteScheduler.from_config(
    loaded_model.scheduler.config, timestep_spacing="trailing"
)

print(f'⏳ Injecting Lightning LoRA...')
loaded_model.load_lora_weights(LORA_MODEL_ID, weight_name="sdxl_lightning_4step_lora.safetensors")
loaded_model.fuse_lora()

# --- THE SPEED REVEAL (Corrected Syntax) ---
loaded_model.disable_attention_slicing()
loaded_model.vae.disable_tiling()
loaded_model.vae.disable_slicing()

print('✅ Engine UNLEASHED. Ready for 1-second bakes.')

In [ ]:
# ==========================================
# 🚀 CELL 3 — LAUNCH STUDIO & TUNNEL (The Definitive Edition)
# ==========================================
# 1. Kill any ghost processes blocking the port
!fuser -k 8000/tcp

import nest_asyncio
import uvicorn
from fastapi import FastAPI, Query
from fastapi.responses import StreamingResponse, HTMLResponse
import io, torch, random, threading, urllib.request, subprocess, time, asyncio, re
import __main__

# Apply nest_asyncio to prevent loop conflicts
nest_asyncio.apply()

app = FastAPI()
gpu_lock = threading.Lock()

# --- SAFER DEVICE DETECTION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- THE FRONTEND (FULL STUDIO UI + FIXED ZEN MODE) ---
html_content = r"""
<!DOCTYPE html>
<html>
<head>
    <title>Vision Studio Feed</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
    <style>
        body, html { margin: 0; padding: 0; height: 100%; background: #000; color: white; font-family: sans-serif; overflow: hidden; }
        #feed { height: 100vh; width: 100vw; overflow-y: scroll; scroll-snap-type: y mandatory; scroll-behavior: smooth; -webkit-overflow-scrolling: touch; }
        .post { height: 100vh; width: 100vw; scroll-snap-align: start; display: flex; justify-content: center; align-items: center; position: relative; background: #050505; overflow: hidden; }
        .post img { height: 100%; width: 100%; object-fit: contain; }
        .loading-text { position: absolute; font-size: 1.2rem; color: #666; font-weight: bold; text-align: center; }

        .post-overlay { position: absolute; right: 18px; bottom: 18px; display: flex; flex-direction: column; gap: 8px; z-index: 4; transition: opacity 0.3s ease; }
        .post-btn { background: rgba(20,20,20,0.88); color: #fff; border: 1px solid #444; border-radius: 999px; padding: 8px 12px; cursor: pointer; font-size: 0.8rem; backdrop-filter: blur(8px); }
        .post-btn:hover { background: rgba(40,40,40,0.92); }

        .post-meta { position: absolute; left: 16px; bottom: 16px; z-index: 3; background: rgba(10,10,10,0.72); border: 1px solid #333; border-radius: 12px; padding: 10px 12px; max-width: min(72vw, 720px); font-size: 0.82rem; line-height: 1.35; color: #ddd; backdrop-filter: blur(10px); transition: opacity 0.3s ease; }
        .post-meta .meta-prompt { display: -webkit-box; -webkit-line-clamp: 3; -webkit-box-orient: vertical; overflow: hidden; text-overflow: ellipsis; }

        #controls { position: fixed; top: 10px; left: 10px; background: rgba(15, 15, 15, 0.95); backdrop-filter: blur(10px); padding: 15px; border-radius: 12px; border: 1px solid #333; z-index: 1000; width: 340px; max-height: 90vh; overflow-y: auto; transition: transform 0.3s ease, opacity 0.3s ease; box-shadow: 0 10px 30px rgba(0,0,0,0.8); }
        #controls.hidden { transform: translateX(-120%); }

        #historyPanel { position: fixed; top: 10px; right: 10px; width: 340px; max-height: 90vh; overflow-y: auto; background: rgba(15, 15, 15, 0.95); backdrop-filter: blur(10px); padding: 15px; border-radius: 12px; border: 1px solid #333; z-index: 1000; transition: transform 0.3s ease, opacity 0.3s ease; box-shadow: 0 10px 30px rgba(0,0,0,0.8); }
        #historyPanel.hidden { transform: translateX(120%); }

        label { font-size: 0.8rem; color: #aaa; margin-top: 10px; display: block; }
        textarea, input[type="text"], input[type="number"], select { width: 100%; background: #111; color: white; border: 1px solid #444; border-radius: 6px; padding: 8px; box-sizing: border-box; margin-top: 4px; font-family: inherit; }
        textarea { height: 64px; resize: none; }
        input[type="range"] { width: 100%; margin-top: 8px; }

        #telemetry, .panel-box { margin-top: 15px; padding: 12px; background: #111; border-radius: 6px; font-family: monospace; font-size: 0.85rem; border: 1px solid #444; color: #0ea5e9; }
        .stat-row { display: flex; justify-content: space-between; margin-bottom: 4px; gap: 10px; }

        details { margin-top: 10px; border: 1px solid #333; border-radius: 6px; padding: 8px; background: #1a1a1a; }
        summary { cursor: pointer; font-size: 0.9rem; font-weight: bold; color: #ddd; }
        .flex-row { display: flex; gap: 10px; margin-top: 10px; }
        .flex-col { flex: 1; }

        .preset-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-top: 8px; }
        .preset-btn, .small-btn, .mode-btn { background: #1b1b1b; color: #ddd; border: 1px solid #444; border-radius: 8px; padding: 8px; cursor: pointer; font-size: 0.82rem; }
        .preset-btn:hover, .small-btn:hover, .mode-btn:hover { background: #262626; }
        .mode-btn.active { border-color: #e11d48; box-shadow: 0 0 0 1px #e11d48 inset; }
        .res-note { font-size: 0.75rem; color: #888; margin-top: 6px; }

        button.primary { width: 100%; padding: 12px; background: #e11d48; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; margin-top: 15px; font-size: 1rem; }
        button.primary:hover { background: #be123c; }
        button.primary:active { transform: scale(0.98); }

        .toggle-ui, .toggle-history, .toggle-zen { position: fixed; bottom: 20px; z-index: 1001; background: #333; border: none; border-radius: 50%; width: 50px; height: 50px; color: white; cursor: pointer; box-shadow: 0px 5px 15px rgba(0,0,0,0.5); font-size: 20px; transition: 0.3s; }
        .toggle-ui { left: 20px; }
        .toggle-history { right: 20px; }
        .toggle-zen { left: 50%; transform: translateX(-50%); background: rgba(255,255,255,0.15); border: 1px solid #555; }
        .toggle-zen:hover { background: #e11d48; border-color: #e11d48; }

        /* --- ZEN MODE LOGIC (FIXED) --- */
        /* Hides panels, eye/clock buttons, and image overlays. Ignores the Ghost button! */
        .zen-active #controls,
        .zen-active #historyPanel,
        .zen-active .toggle-ui,
        .zen-active .toggle-history,
        .zen-active .post-overlay,
        .zen-active .post-meta {
            opacity: 0 !important;
            pointer-events: none !important;
        }

        .history-item { border: 1px solid #333; background: #111; border-radius: 10px; padding: 10px; margin-top: 10px; }
        .history-item img { width: 100%; aspect-ratio: 3/4; object-fit: cover; border-radius: 8px; border: 1px solid #333; background: #050505; cursor: pointer; }
        .history-actions { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-top: 8px; }
        .history-text { font-size: 0.78rem; color: #ddd; margin-top: 8px; line-height: 1.35; }
        .badge { display: inline-block; padding: 2px 8px; border-radius: 999px; font-size: 0.72rem; background: #222; color: #bbb; border: 1px solid #3a3a3a; }
        .fav { color: #ffcc5a; }
        .checkbox-row { display: flex; align-items: center; gap: 8px; color: white; font-size: 0.9rem; margin-top: 10px; cursor: pointer; }
        .checkbox-row input { width: auto; }
        .section-title { font-size: 0.8rem; color: #aaa; margin-top: 10px; }
    </style>
</head>
<body id="mainBody">
    <button class="toggle-ui" onclick="toggleUI()">👁️</button>
    <button class="toggle-history" onclick="toggleHistory()">🕘</button>
    <button class="toggle-zen" onclick="toggleZen(event)" title="Zen Mode">👻</button>

    <div id="controls">
        <h2 style="margin: 0 0 10px 0; font-size: 1.2rem;">🔴 Vision Studio</h2>

        <div id="telemetry">
            <div class="stat-row"><span>Buffer Ready:</span> <span id="buf_count">0</span> / <span id="buf_target">3</span></div>
            <div class="stat-row"><span>Jobs Active:</span> <span id="active_jobs">0</span> / <span id="max_jobs">2</span></div>
            <div class="stat-row"><span>Current Time:</span> <span id="current_timer">0.00s</span></div>
            <div class="stat-row"><span>Avg Time:</span> <span id="avg_timer">0.00s</span></div>
            <div class="stat-row"><span>Engine State:</span> <span id="engine_state" style="color: #94a3b8;">⏸️ Idle</span></div>
        </div>

        <div class="section-title">Mode</div>
        <div class="preset-grid">
            <button type="button" class="mode-btn active" id="mode_balanced" onclick="applyMode('balanced')">Balanced</button>
            <button type="button" class="mode-btn" id="mode_fast" onclick="applyMode('fast')">Fast</button>
            <button type="button" class="mode-btn" id="mode_quality" onclick="applyMode('quality')">Quality</button>
        </div>

        <label>Preload Buffer Size: <span id="buffer_val">3</span> frames</label>
        <input type="range" id="buffer_size" min="1" max="10" value="3" oninput="document.getElementById('buffer_val').innerText = this.value; maintainBuffer();">

        <label>Subject Prompt</label>
        <textarea id="prompt">masterpiece, best quality, newest</textarea>

        <div class="preset-grid">
            <button type="button" class="small-btn" onclick="applyPromptPreset('anime_clean')">Anime Clean</button>
            <button type="button" class="small-btn" onclick="applyPromptPreset('cinematic')">Cinematic</button>
            <button type="button" class="small-btn" onclick="applyPromptPreset('soft')">Soft Lighting</button>
            <button type="button" class="small-btn" onclick="applyPromptPreset('detail')">Detail Boost</button>
        </div>

        <div class="preset-grid">
            <button type="button" class="small-btn" onclick="mutatePrompt('detail')">+ Detail</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('dramatic')">+ Dramatic</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('pose')">+ Pose</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('camera')">+ Camera</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('style')">+ Style</button>
            <button type="button" class="small-btn" onclick="mutatePrompt('random')">+ Random</button>
        </div>

        <label style="display: flex; align-items: center; gap: 8px; color: white; font-size: 0.9rem; margin-top: 10px; cursor: pointer;">
            <input type="checkbox" id="grunge" style="width: auto;"> 🔥 Apply Glitch/Grunge
        </label>

        <label class="checkbox-row">
            <input type="checkbox" id="infinite_mode"> ♾️ Infinite Mode
        </label>

        <label class="checkbox-row">
            <input type="checkbox" id="seed_lock"> 🔒 Seed Lock
        </label>

        <div class="flex-row">
            <div class="flex-col">
                <label>Seed</label>
                <input type="number" id="seed" value="">
            </div>
            <div class="flex-col" style="display:flex;align-items:flex-end;">
                <button type="button" class="small-btn" style="width:100%;" onclick="rerollSeed()">🎲 Reroll</button>
            </div>
        </div>

        <details>
            <summary>⚙️ Advanced Parameters</summary>
            <label>Negative Prompt</label>
            <textarea id="neg_prompt" style="height: 40px;">worst quality, low quality, normal quality, blurry, bad anatomy</textarea>

            <div class="preset-grid">
                <button type="button" class="small-btn" onclick="applyNegativePreset('default')">Default Neg</button>
                <button type="button" class="small-btn" onclick="applyNegativePreset('clean')">Cleaner Faces</button>
                <button type="button" class="small-btn" onclick="applyNegativePreset('sharp')">Sharp Output</button>
                <button type="button" class="small-btn" onclick="applyNegativePreset('nsfw')">NSFW Detail</button>
            </div>

            <label>Inference Steps: <span id="steps_val">4</span></label>
            <input type="range" id="steps" min="1" max="12" value="4" oninput="document.getElementById('steps_val').innerText = this.value">

            <label>CFG Scale: <span id="cfg_val">1.5</span></label>
            <input type="range" id="cfg" min="0.5" max="6.0" step="0.1" value="1.5" oninput="document.getElementById('cfg_val').innerText = this.value">

            <label>Resolution Presets</label>
            <div class="preset-grid">
                <button type="button" class="preset-btn" onclick="setRes(480, 720)">Default — 480×720</button>
                <button type="button" class="preset-btn" onclick="setRes(512, 512)">1:1 — 512×512</button>
                <button type="button" class="preset-btn" onclick="setRes(768, 1024)">3:4 — 768×1024</button>
                <button type="button" class="preset-btn" onclick="setRes(832, 1216)">SDXL — 832×1216</button>
                <button type="button" class="preset-btn" onclick="setRes(1216, 832)">Wide — 1216×832</button>
                <button type="button" class="preset-btn" onclick="setRes(1024, 1536)">Portrait — 1024×1536</button>
            </div>

            <div class="flex-row">
                <div class="flex-col">
                    <label>Width</label>
                    <input type="number" id="width" value="480" min="256" max="2048" step="16" oninput="handleDimensionChange('width')">
                </div>
                <div class="flex-col">
                    <label>Height</label>
                    <input type="number" id="height" value="720" min="256" max="2048" step="16" oninput="handleDimensionChange('height')">
                </div>
            </div>

            <label class="checkbox-row">
                <input type="checkbox" id="aspect_lock" checked> 🔗 Aspect Lock
            </label>

            <label class="checkbox-row">
                <input type="checkbox" id="vram_safe" checked> 🛡️ VRAM Safe Mode
            </label>
        </details>

        <button class="primary" onclick="igniteEngine()">Ignite Feed ▶</button>
        <button class="primary" style="background:#333;" onclick="stopInfinite()">Stop Infinite ⏹</button>
    </div>

    <div id="historyPanel" class="hidden">
        <h2 style="margin: 0 0 10px 0; font-size: 1.1rem;">🕘 History</h2>
        <div class="panel-box">
            <div class="stat-row"><span>Saved Frames:</span><span id="history_count">0</span></div>
            <div class="stat-row"><span>Favorites:</span><span id="favorite_count">0</span></div>
        </div>
        <div id="historyList"></div>
    </div>

    <div id="feed" onscroll="handleScroll()"></div>

    <script>
        let activeJobs = 0;
        const MAX_ACTIVE_JOBS = 2;
        const DOM_PRUNE_LIMIT = 25;
        let generationDurations = [];
        let liveTimers = {};
        let historyItems = [];
        let lastSeedUsed = null;
        let aspectRatio = 480 / 720;
        let infiniteStopped = false;

        const promptPresets = {
            anime_clean: "masterpiece, best quality, newest, clean lineart, polished anime shading, expressive eyes",
            cinematic: "masterpiece, best quality, newest, cinematic lighting, dramatic shadows, filmic composition, dynamic framing",
            soft: "masterpiece, best quality, newest, soft lighting, pastel glow, delicate atmosphere, gentle highlights",
            detail: "masterpiece, best quality, newest, ultra detailed, intricate details, refined textures, crisp focus"
        };
        const negativePresets = {
            default: "worst quality, low quality, normal quality, blurry, bad anatomy",
            clean: "worst quality, low quality, blurry, cross-eye, deformed eyes, bad anatomy, bad hands, extra fingers",
            sharp: "low quality, blurry, soft focus, foggy details, smudged lines, jpeg artifacts",
            nsfw: "worst quality, low quality, blurry, deformed anatomy, bad hands, extra limbs, duplicate body parts"
        };
        const mutations = {
            detail: "intricate details, refined textures, sharp focus",
            dramatic: "dramatic lighting, intense contrast, striking composition",
            pose: "dynamic pose, expressive body language, natural gesture",
            camera: "low angle shot, cinematic framing, depth of field",
            style: "stylized rendering, polished finish, premium anime aesthetic",
            random: ["wind-swept hair, motion energy", "golden hour lighting, warm glow", "rainy atmosphere, reflective surfaces", "neon rim light, night city vibe", "soft bloom, dreamy mood", "heroic stance, centered framing"]
        };

        function toggleUI() { document.getElementById('controls').classList.toggle('hidden'); }
        function toggleHistory() { document.getElementById('historyPanel').classList.toggle('hidden'); }

        function toggleZen(e) {
            if(e) e.stopPropagation();
            document.body.classList.toggle('zen-active');
        }

        document.addEventListener('click', (e) => {
            if (document.body.classList.contains('zen-active')) { toggleZen(); }
        });
        document.getElementById('controls').addEventListener('click', (e) => e.stopPropagation());
        document.getElementById('historyPanel').addEventListener('click', (e) => e.stopPropagation());

        function setModeButton(mode) {
            ["fast", "balanced", "quality"].forEach(m => { document.getElementById("mode_" + m).classList.remove("active"); });
            document.getElementById("mode_" + mode).classList.add("active");
        }

        function applyMode(mode) {
            setModeButton(mode);
            if (mode === "fast") { document.getElementById("steps").value = 3; document.getElementById("steps_val").innerText = "3"; document.getElementById("cfg").value = 1.2; document.getElementById("cfg_val").innerText = "1.2"; setRes(480, 720); }
            else if (mode === "balanced") { document.getElementById("steps").value = 4; document.getElementById("steps_val").innerText = "4"; document.getElementById("cfg").value = 1.5; document.getElementById("cfg_val").innerText = "1.5"; setRes(480, 720); }
            else if (mode === "quality") { document.getElementById("steps").value = 6; document.getElementById("steps_val").innerText = "6"; document.getElementById("cfg").value = 2.0; document.getElementById("cfg_val").innerText = "2.0"; setRes(768, 1024); }
        }

        function applyPromptPreset(kind) { document.getElementById("prompt").value = promptPresets[kind] || document.getElementById("prompt").value; }
        function applyNegativePreset(kind) { document.getElementById("neg_prompt").value = negativePresets[kind] || document.getElementById("neg_prompt").value; }

        function mutatePrompt(kind) {
            const promptBox = document.getElementById("prompt");
            let extra = "";
            if (kind === "random") { const arr = mutations.random; extra = arr[Math.floor(Math.random() * arr.length)]; }
            else { extra = mutations[kind] || ""; }
            if (!extra) return;
            promptBox.value = promptBox.value.trim().replace(/,\s*$/, "");
            promptBox.value += ", " + extra;
        }

        function rerollSeed() { const seed = Math.floor(Math.random() * 2147483647); document.getElementById("seed").value = seed; lastSeedUsed = seed; }

        function setRes(w, h) { document.getElementById('width').value = w; document.getElementById('height').value = h; aspectRatio = w / h; applyVramSafeIfNeeded(); }
        function roundTo16(v) { return Math.max(256, Math.round(v / 16) * 16); }

        function handleDimensionChange(changed) {
            let w = parseInt(document.getElementById('width').value) || 480;
            let h = parseInt(document.getElementById('height').value) || 720;
            if (document.getElementById("aspect_lock").checked) {
                if (changed === "width") { h = roundTo16(w / aspectRatio); document.getElementById('height').value = h; }
                else { w = roundTo16(h * aspectRatio); document.getElementById('width').value = w; }
            } else { if (w > 0 && h > 0) aspectRatio = w / h; }
            applyVramSafeIfNeeded();
        }

        function getSafeRes() {
            let w = parseInt(document.getElementById('width').value) || 480;
            let h = parseInt(document.getElementById('height').value) || 720;
            w = Math.max(256, Math.min(w, 2048)); h = Math.max(256, Math.min(h, 2048)); w = roundTo16(w); h = roundTo16(h);
            if (document.getElementById("vram_safe").checked) {
                const maxPixels = 786432;
                if (w * h > maxPixels) { const scale = Math.sqrt(maxPixels / (w * h)); w = roundTo16(w * scale); h = roundTo16(h * scale); }
            }
            return { w, h };
        }

        function applyVramSafeIfNeeded() { const { w, h } = getSafeRes(); document.getElementById('width').value = w; document.getElementById('height').value = h; }

        function getSelectedSeed() {
            const lock = document.getElementById("seed_lock").checked;
            const seedInput = document.getElementById("seed");
            let seedVal = seedInput.value.trim();
            if (lock) {
                if (!seedVal) { seedVal = String(Math.floor(Math.random() * 2147483647)); seedInput.value = seedVal; }
                return parseInt(seedVal);
            }
            return -1;
        }

        function updateTelemetry() {
            const feed = document.getElementById('feed');
            const currentIndex = feed.children.length > 0 ? Math.round(feed.scrollTop / window.innerHeight) : 0;
            const imagesAhead = Math.max(0, feed.children.length - currentIndex - 1);
            const targetBuffer = parseInt(document.getElementById('buffer_size').value);

            document.getElementById('buf_count').innerText = imagesAhead;
            document.getElementById('buf_target').innerText = targetBuffer;
            document.getElementById('active_jobs').innerText = activeJobs;
            document.getElementById('max_jobs').innerText = MAX_ACTIVE_JOBS;

            let currentLive = 0;
            for (const key in liveTimers) { currentLive = Math.max(currentLive, (performance.now() - liveTimers[key]) / 1000); }
            document.getElementById("current_timer").innerText = currentLive.toFixed(2) + "s";

            const avg = generationDurations.length ? generationDurations.reduce((a,b)=>a+b,0) / generationDurations.length : 0;
            document.getElementById("avg_timer").innerText = avg.toFixed(2) + "s";

            const engineState = document.getElementById('engine_state');
            if (activeJobs > 0) engineState.innerHTML = '<span style="color: #ef4444;">🔴 Baking...</span>';
            else if (document.getElementById("infinite_mode").checked && !infiniteStopped) engineState.innerHTML = '<span style="color: #22c55e;">♾️ Infinite Ready</span>';
            else engineState.innerHTML = '<span style="color: #22c55e;">🟢 Ready</span>';

            document.getElementById("history_count").innerText = historyItems.length;
            document.getElementById("favorite_count").innerText = historyItems.filter(x => x.favorite).length;
            return { imagesAhead, targetBuffer };
        }

        function copyText(text) { navigator.clipboard.writeText(text).catch(() => {}); }
        function buildSettingsText(item) { return JSON.stringify({ prompt: item.prompt, negative_prompt: item.neg_prompt, steps: item.steps, cfg: item.cfg, width: item.width, height: item.height, seed: item.seed, grunge: item.grunge }, null, 2); }
        function downloadBlobUrl(url, filename="vision_frame.jpeg") { const a = document.createElement("a"); a.href = url; a.download = filename; document.body.appendChild(a); a.click(); a.remove(); }

        function rerunFromHistory(item) {
            document.getElementById("prompt").value = item.prompt; document.getElementById("neg_prompt").value = item.neg_prompt;
            document.getElementById("steps").value = item.steps; document.getElementById("steps_val").innerText = item.steps;
            document.getElementById("cfg").value = item.cfg; document.getElementById("cfg_val").innerText = item.cfg;
            document.getElementById("grunge").checked = item.grunge;
            document.getElementById("seed_lock").checked = true; document.getElementById("seed").value = item.seed;
            setRes(item.width, item.height); igniteEngine();
        }

        function renderHistory() {
            const wrap = document.getElementById("historyList"); wrap.innerHTML = "";
            historyItems.slice().reverse().forEach((item, idx) => {
                const realIndex = historyItems.length - 1 - idx;
                const div = document.createElement("div"); div.className = "history-item";
                div.innerHTML = `<img src="${item.url}" alt="history" onclick="window.open('${item.url}')"><div class="history-text"><div style="display:flex;justify-content:space-between;align-items:center;gap:8px;"><span class="badge">${item.width}×${item.height}</span><span class="badge">${item.steps} steps</span><span class="badge">CFG ${item.cfg}</span><span class="badge">Seed ${item.seed}</span></div><div style="margin-top:8px;"><span class="${item.favorite ? 'fav' : ''}">${item.favorite ? '★ Favorite' : '☆ Not Favorite'}</span></div><div style="margin-top:8px;">${item.prompt}</div></div><div class="history-actions"><button class="small-btn" onclick="toggleFavorite(${realIndex})">${item.favorite ? '★ Unfavorite' : '☆ Favorite'}</button><button class="small-btn" onclick="downloadBlobUrl('${item.url}', 'vision_history_${realIndex}.jpeg')">⬇ Download</button><button class="small-btn" onclick='copyText(historyItems[${realIndex}].prompt)'>📋 Prompt</button><button class="small-btn" onclick='copyText(buildSettingsText(historyItems[${realIndex}]))'>⚙ Settings</button><button class="small-btn" onclick='rerunFromHistory(historyItems[${realIndex}])'>🔁 Rerun</button><button class="small-btn" onclick='copyText(historyItems[${realIndex}].neg_prompt)'>🧪 Neg</button></div>`;
                wrap.appendChild(div);
            });
            updateTelemetry();
        }

        function toggleFavorite(index) { historyItems[index].favorite = !historyItems[index].favorite; renderHistory(); }

        function attachPostActions(post, item) {
            const overlay = document.createElement("div"); overlay.className = "post-overlay";
            const favBtn = document.createElement("button"); favBtn.className = "post-btn"; favBtn.textContent = item.favorite ? "★ Favorite" : "☆ Favorite";
            favBtn.onclick = () => { item.favorite = !item.favorite; favBtn.textContent = item.favorite ? "★ Favorite" : "☆ Favorite"; renderHistory(); };
            const dlBtn = document.createElement("button"); dlBtn.className = "post-btn"; dlBtn.textContent = "⬇ Download";
            dlBtn.onclick = () => downloadBlobUrl(item.url, `vision_frame_${item.seed}.jpeg`);
            const cpBtn = document.createElement("button"); cpBtn.className = "post-btn"; cpBtn.textContent = "📋 Prompt";
            cpBtn.onclick = () => copyText(item.prompt);
            const csBtn = document.createElement("button"); csBtn.className = "post-btn"; csBtn.textContent = "⚙ Settings";
            csBtn.onclick = () => copyText(buildSettingsText(item));
            overlay.appendChild(favBtn); overlay.appendChild(dlBtn); overlay.appendChild(cpBtn); overlay.appendChild(csBtn); post.appendChild(overlay);

            const meta = document.createElement("div"); meta.className = "post-meta";
            meta.innerHTML = `<div class="meta-prompt">${item.prompt}</div><div style="margin-top:6px;color:#aaa;">${item.width}×${item.height} • ${item.steps} steps • CFG ${item.cfg} • Seed ${item.seed}</div>`;
            post.appendChild(meta);
        }

        async function bakeNext() {
            if (activeJobs >= MAX_ACTIVE_JOBS) return;
            activeJobs += 1;
            const jobId = "job_" + Date.now() + "_" + Math.random().toString(36).slice(2);
            liveTimers[jobId] = performance.now();
            updateTelemetry();

            const promptRaw = document.getElementById('prompt').value;
            const negPromptRaw = document.getElementById('neg_prompt').value;
            const prompt = encodeURIComponent(promptRaw);
            const neg_prompt = encodeURIComponent(negPromptRaw);
            const grunge = document.getElementById('grunge').checked;
            const steps = document.getElementById('steps').value;
            const cfg = document.getElementById('cfg').value;
            const { w, h } = getSafeRes();
            const seed = getSelectedSeed();

            const post = document.createElement('div');
            post.className = 'post'; post.innerHTML = '<div class="loading-text">Baking Frame...</div>';
            const feed = document.getElementById('feed'); feed.appendChild(post);

            try {
                const url = `/api/generate?prompt=${prompt}&neg_prompt=${neg_prompt}&grunge=${grunge}&steps=${steps}&cfg=${cfg}&width=${w}&height=${h}&seed=${seed}&_t=${Date.now()}`;
                const response = await fetch(url, { cache: 'no-store' });
                if (!response.ok) throw new Error('Failed');

                const blob = await response.blob();
                const imgUrl = URL.createObjectURL(blob);
                const usedSeed = parseInt(response.headers.get("X-Seed") || seed || 0);

                lastSeedUsed = usedSeed;
                if (document.getElementById("seed_lock").checked) document.getElementById("seed").value = usedSeed;

                const img = document.createElement("img");
                img.src = imgUrl; img.loading = "eager"; img.decoding = "async"; img.ondblclick = () => window.open(imgUrl, '_blank');
                post.innerHTML = ""; post.appendChild(img);

                const item = { url: imgUrl, prompt: promptRaw, neg_prompt: negPromptRaw, steps: parseInt(steps), cfg: parseFloat(cfg), width: w, height: h, seed: usedSeed, grunge: grunge, favorite: false };
                historyItems.push(item); renderHistory(); attachPostActions(post, item);

                const duration = (performance.now() - liveTimers[jobId]) / 1000;
                generationDurations.push(duration);
                if (generationDurations.length > 20) generationDurations.shift();
            } catch (err) { post.innerHTML = '<div class="loading-text" style="color: #e11d48;">⚠️ Engine Stall</div>'; }
            finally {
                delete liveTimers[jobId];
                if (feed.children.length > DOM_PRUNE_LIMIT) {
                    const currentIndex = Math.round(feed.scrollTop / window.innerHeight);
                    if (currentIndex > 5) feed.removeChild(feed.firstChild);
                }
                activeJobs -= 1;
                maintainBuffer();
                if (document.getElementById("infinite_mode").checked && !infiniteStopped) setTimeout(() => maintainBuffer(), 10);
            }
        }

        function maintainBuffer() {
            const stats = updateTelemetry();
            while ((stats.imagesAhead + activeJobs) < stats.targetBuffer && activeJobs < MAX_ACTIVE_JOBS) { bakeNext(); }
        }

        function handleScroll() { maintainBuffer(); }
        function igniteEngine() { infiniteStopped = false; document.getElementById('feed').innerHTML = ''; activeJobs = 0; liveTimers = {}; maintainBuffer(); }
        function stopInfinite() { infiniteStopped = true; document.getElementById("infinite_mode").checked = false; updateTelemetry(); }

        setInterval(updateTelemetry, 200);
        applyMode('balanced'); applyNegativePreset('default');
        document.getElementById("cfg").value = 1.5; document.getElementById("cfg_val").innerText = "1.5";
        setRes(480, 720);
    </script>
</body>
</html>
"""

# --- THE BACKEND (FASTAPI + GPU) ---
@app.get("/")
def home():
    return HTMLResponse(content=html_content)

@app.get("/api/generate")
async def generate_api(
    prompt: str,
    neg_prompt: str = "worst quality, low quality",
    grunge: str = "false",
    steps: int = 4,
    cfg: float = 1.5,
    width: int = 480,
    height: int = 720,
    seed: int = Query(-1)
):
    model = getattr(__main__, 'loaded_model', None)
    if not model:
        return {"error": "Model missing - Run Cell 2"}

    if grunge.lower() == "true":
        prompt = f'{prompt}, glitch art, grunge style, abstract horror, chromatic aberration, striking composition'

    full_prompt = f'masterpiece, best quality, newest, absurdres, {prompt}'

    with gpu_lock:
        used_seed = seed if seed is not None and int(seed) >= 0 else random.randint(0, 2147483647)
        generator = torch.Generator(device=DEVICE).manual_seed(int(used_seed))

        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.float16):
            result = model(
                prompt=full_prompt,
                negative_prompt=neg_prompt,
                num_inference_steps=steps,
                guidance_scale=cfg,
                width=width,
                height=height,
                generator=generator
            )

    img_byte_arr = io.BytesIO()
    result.images[0].save(img_byte_arr, format="JPEG", quality=90)
    img_byte_arr.seek(0)

    return StreamingResponse(
        img_byte_arr,
        media_type="image/jpeg",
        headers={
            "Cache-Control": "no-store, no-cache, must-revalidate, max-age=0",
            "X-Seed": str(used_seed)
        }
    )

# --- THE LAUNCHER (CLOUDFLARE SETUP) ---
PORT = 8000
print("🌐 [1/2] Establishing Secure Cloudflare Tunnel...")

with open("tunnel.log", "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stdout=f, stderr=f)

url = ""
for _ in range(30): # Scan for 15 seconds
    time.sleep(0.5)
    try:
        with open("tunnel.log", "r") as f:
            log_content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
            if match:
                url = match.group(0)
                break
    except: pass

print("="*60)
if url:
    print(f"🔗 STUDIO LINK: {url}")
    print("="*60)
    print("If Cloudflare gives a security warning, simply click 'Visit Site'.")
else:
    print("⚠️ Cloudflare is taking too long. Check tunnel.log.")
print("="*60)
print("🛑 STARTING LOCAL SERVER... THE CELL IS NOW LOCKED AND FULLY ACTIVE.")

# --- NATIVE SERVER KEEP-ALIVE ---
config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="error")
server = uvicorn.Server(config)
await server.serve()